# 4.3 GEMM 的流水优化

## 概述

上一节已经实现基础 GEMM：8 个核任务各自循环处理多个输出块，每轮完成输入搬入、Cube 计算和结果搬出。本节沿用用例 `(512,36864,256)`，让后续输出块的搬运与当前块的计算并行，再用相同输入验证多 buffer 的性能效果。

## 学习目标

- 理解 Cube 中通过流水并行进行性能优化的原理。
- 在同一 8 核用例上对比单、双、三 buffer 的正确性与 kernel 耗时。
- 计算实际算力和数据搬运量，分析性能上限及可能的瓶颈。

[本章导航](README.md) · [上一节](04.02_basic_gemm.ipynb) · [下一节](04.04_persistent.ipynb)

<p><small>执行前完成第1.4节环境准备，按本章README的统一入口运行。每页使用新内核；NPU程序由独立子进程执行，notebook内核不初始化设备。</small></p>

## 1. 多 buffer 流水并行

基础版中 a、b、c 各有一份内存。下一输出块复用 a/b 前，要等待当前输入读完；复用 c 前，要等待当前结果写回。双 buffer 为这些数据分别准备两份内存，让不同输出块处于不同流水。

<img alt="核内输出块的多 buffer 流水" src="images/04.03_output_pipeline.svg" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

每轮都是一个独立输出块。第 0、2、4…轮使用内存 0，第 1、3、5…轮使用内存 1；输入就绪后才能计算，结果写完后对应 L0C 内存才能复用。图中的交叠表示依赖允许的执行关系，实际耗时由 msprof 测量。

<p><small>工具说明：msprof 是昇腾的性能采集工具，用于获取算子在设备上的执行耗时等信息。其详细用法和性能分析方法将在第6章系统介绍。本章先按给定命令完成采集，读取 kernel 耗时并比较优化前后的结果，无需提前掌握全部参数。</small></p>

### 从基础版到多 buffer

保持 4.2 中的索引、搬运、`T.gemm` 和输出写回不变，同时设置输出块循环的 `num_stages` 与 buffer 数量：

```python
T.annotate_buffer_versions({a: stages, b: stages, c: stages})

for it in T.Pipelined(tasks // 8, num_stages=stages):
    idx = bx * (tasks // 8) + it
    mi = idx // (N // BN)
    ni = idx % (N // BN)
    T.copy(A[mi * BM : (mi + 1) * BM, :], a)
    T.copy(B[ni * BN : (ni + 1) * BN, :], b)
    T.gemm(a, b, c, transpose_B=True, clear_accum=True)

    T.copy(c, C[mi * BM : (mi + 1) * BM, ni * BN : (ni + 1) * BN])
```

`stages=1` 为基础版，2、3 为多 buffer 配置。[完整实现](src/gemm_kernel.py) 中，显式配置 a、b、c 的 buffer 数量，后端生成对应的内存偏移与同步。当前生成代码可看到双 buffer 的 `it & 1` 内存偏移：不仅 a/b 有两份，c 也有两份。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">stages</th>
<th style="text-align: left; vertical-align: top;">a/b 合计 L1</th>
<th style="text-align: left; vertical-align: top;">c 合计 L0C</th>
<th style="text-align: left; vertical-align: top;">每核输出轮数</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">128 KiB</td>
<td style="text-align: left; vertical-align: top;">64 KiB</td>
<td style="text-align: left; vertical-align: top;">144</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">2</td>
<td style="text-align: left; vertical-align: top;">256 KiB</td>
<td style="text-align: left; vertical-align: top;">128 KiB</td>
<td style="text-align: left; vertical-align: top;">144</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">3</td>
<td style="text-align: left; vertical-align: top;">384 KiB</td>
<td style="text-align: left; vertical-align: top;">192 KiB</td>
<td style="text-align: left; vertical-align: top;">144</td>
</tr>
</tbody>
</table>

本节通过 `T.annotate_buffer_versions` 显式设置 buffer 份数，便于把代码与容量表对应。4.5 延续同样的写法。



**L1 和 L0C 必须分别检查。**

增加 buffer 会消耗更多内存，且流水填充、排空和同步仍有开销；不能仅凭 stages 更大判断更快。

## 2. 性能对比验证

固定 `M/N/K=512/36864/256`，`BM=BN=128`、8 核、bf16 输入和 fp32 输出，将循环 stages 与 a/b/c 的 buffer 数量同时设为 1、2、3。每核处理 144 个完整输出 tile。

先用随机、全零、常数和带符号周期输入验证全部输出，每类连续调用两次；再复用同一输入采集三批，每批预热 10 次、正式采样 30 次。msprof `op_summary` 中筛选 `main_kernel`、`AI_CORE`、Block Num=8，读取 `Task Duration(us)`。剔除 8 次精度调用和每批预热，保留 90 个有效样本。

kernel 耗时包含设备上的搬运、计算和同步，不含 Python、编译和 Host 等待。输入复用，不主动清缓存。

```bash
source src/env.sh
msprof --output="${COURSE_OUTPUT_DIR}/profile_s2" --task-time=on --ai-core=off --runtime-api=on \
  python src/profile_gemm.py 512 36864 256 2 contiguous > "${COURSE_OUTPUT_DIR}/profile_s2.log" 2>&1
python src/summarize_profile.py "${COURSE_OUTPUT_DIR}/profile_s2" --blocks 8 --log "${COURSE_OUTPUT_DIR}/profile_s2.log"
```

[采集入口](src/profile_gemm.py)与[样本核对程序](src/summarize_profile.py)随文交付。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">buffer 数量</th>
<th style="text-align: left; vertical-align: top;">三批中位耗时（μs）</th>
<th style="text-align: left; vertical-align: top;">kernel 中位耗时（μs）</th>
<th style="text-align: left; vertical-align: top;">相对单 buffer 加速比</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">225.4760 / 219.5210 / 219.2795</td>
<td style="text-align: left; vertical-align: top;">219.9225</td>
<td style="text-align: left; vertical-align: top;">1.00×</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">2</td>
<td style="text-align: left; vertical-align: top;">132.1335 / 132.0415 / 131.8640</td>
<td style="text-align: left; vertical-align: top;">132.0520</td>
<td style="text-align: left; vertical-align: top;">1.67×</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">3</td>
<td style="text-align: left; vertical-align: top;">123.3620 / 123.4760 / 123.1915</td>
<td style="text-align: left; vertical-align: top;">123.3605</td>
<td style="text-align: left; vertical-align: top;">1.78×</td>
</tr>
</tbody>
</table>

历史采集时四类输入全部通过，最大绝对误差为 2.3841858e-06。课程目录未附历史原始样本；下方代码重新验证正确性，性能请按本节命令重新采集。每次增加 buffer 都需要重新测量，数量更多不保证更快。

In [1]:
import subprocess, sys
p = subprocess.run([sys.executable, 'src/run_gemm.py', 'pipeline'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print("\n".join(line for line in p.stdout.splitlines()
                       if not line.startswith("Loading tilelang libs from dev root:")))
p.check_returncode()

[W917 20:45:19.598192066 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
512_36864_256_128_128_s1_p0 random: full-output PASS; shape=(512, 36864); max_abs_error=2.3841858e-06
512_36864_256_128_128_s1_p0 zero: full-output PASS; shape=(512, 36864); max_abs_error=0
512_36864_256_128_128_s1_p0 constant: full-output PASS; shape=(512, 36864); max_abs_error=0
512_36864_256_128_128_s1_p0 pattern: full-output PASS; shape=(512, 36864); max_abs_error=0
2026-09-17 20:45:29  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `main` with `out_idx=[2]`
2026-09-17 20:45:41  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `main`
512_36864_256_128_128_s2_p0 random: full-output PASS; shape=(512, 36864); max_abs_error=2.3841858e-06
512_36864_256_128_128_s2_p0 zero: full-output PASS; shape=(512, 36864); max_abs_error=0
512_36864_256_128_128_s2_p

## 3. 性能瓶颈分析

以 msprof 的 kernel 耗时 t（单位 μs）计算下面三个指标：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">指标</th>
<th style="text-align: left; vertical-align: top;">计算方式</th>
<th style="text-align: left; vertical-align: top;">备注：各项意义与单位</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">实际算力（TFLOPS）</td>
<td style="text-align: left; vertical-align: top;"><code>F / (t×10^6)</code></td>
<td style="text-align: left; vertical-align: top;">计算量 / 耗时。<code>F=2MNK</code>，单位 FLOPs；t 为 kernel 耗时，单位 μs；结果单位为 10^12 FLOPs/s。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">有效带宽（GB/s）</td>
<td style="text-align: left; vertical-align: top;"><code>Q / (t×1000)</code></td>
<td style="text-align: left; vertical-align: top;">有效数据量 / 耗时。<code>Q=2MK+2NK+4MN</code>，单位字节；t 单位 μs；结果单位为 10^9 字节/s。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">算力利用率（%）</td>
<td style="text-align: left; vertical-align: top;"><code>P_actual / P_peak × 100%</code></td>
<td style="text-align: left; vertical-align: top;">实际算力 / 理论峰值。两者均用 TFLOPS；峰值须对应相同数据类型、参与核数和频率口径。</td>
</tr>
</tbody>
</table>

M、N 分别为 A、B 的行数，K 为共同的归约长度，均为元素个数。一次乘法加一次加法计 2 FLOPs；A/B 的 bf16 元素各占 2 字节，C 的 fp32 元素占 4 字节。本例计算量为 9,663,676,416 FLOPs，输入各读一次、结果写一次的数据量为 94,633,984 字节。

带宽和算力利用率用来评价优化效果，对应的数据搬运耗时、计算耗时与总耗时之间的差距用于确定当前性能瓶颈和优化方向。搬运与计算可能并行，分析时需要结合各流水的实测耗时和未被掩盖的等待时间，总耗时并不一定等于各流水耗时的简单相加。

## 小结

本节通过多 buffer 排布输入搬运、Cube 计算和结果写回，并用同一用例验证性能。增加 buffer 后要重新检查 L1/L0C 占用，再结合 kernel 耗时判断收益。下一节学习 Persistent 的任务遍历。

## 课后练习

1. 根据实测表，计算双 buffer 相对单 buffer 的耗时降幅。
2. 使用双 buffer 耗时计算有效带宽和算力利用率。为便于手算，本题给定同一 bf16 配置的 8 核峰值估算值 `117.9648 TFLOPS`；说明为什么不能使用整卡 36 核峰值。该数值仅是本题所用估算口径，实际设备应核对型号、频率与参与核数。

In [2]:
from pathlib import Path
print(Path("answer/04.03_answer.txt").read_text())

将表中 t1、t2 代入：(t1-t2)/t1×100% 得到耗时降幅。
有效带宽=94,633,984/(t2×1000) GB/s；实际算力=9,663,676,416/(t2×10^6) TFLOPS；利用率=实际算力/117.9648×100%。
本节只有8核参与，必须使用8核配置峰值；4.5使用36核时再改为530.8416 TFLOPS。

